In [0]:
from pyspark.sql import functions as F
catalog = spark.sql(
     "SELECT current_catalog()"
).first()[0]

schema = "migration"
df_customers = spark.table(f"{catalog}.{schema}.customers_bronze")
df_orders = spark.table(f"{catalog}.{schema}.orders_bronze")
df_products = spark.table(f"{catalog}.{schema}.products_bronze")
df_transactions = spark.table(f"{catalog}.{schema}.transactions_bronze")

In [0]:
#Customer Transformation 

#step 1: Remove uneccessary spaces 
text_columns=["customer_id","full_name","email","phone","registration_date","account_type","loyalty_tier","marketing_opt_in"]

for c in text_columns:
    df_customers = df_customers.withColumn(c, F.when(F.trim(F.col(c)) == "", F.lit(None)).otherwise(F.trim(F.col(c))))

# step 2: remove ID prefixes
df_customers= df_customers.withColumn("customer_id_cleaned", F.regexp_replace("customer_id", r"^[A-Za-z]", ""))

improper_id= df_customers.filter(~F.col("customer_id_cleaned").rlike(r"^\d+$")).count()
distinct_ids_before = df_customers.select("customer_id").distinct().count()
distinct_ids_after = df_customers.select("customer_id_cleaned").distinct().count()

assert improper_id == 0,"Some IDs are not digits only"
assert distinct_ids_before == distinct_ids_after,"Removing the prefix made two different customer IDs identical."

df_customers = df_customers.drop("customer_id").withColumnRenamed("customer_id_cleaned","customer_id")

# step 3: casing
df_customers = (df_customers.withColumn("full_name", F.initcap("full_name"))
                .withColumn("email",F.lower("email"))
                .withColumn ("account_type",F.initcap("account_type"))
                .withColumn ("loyalty_tier",F.when(F.lower("loyalty_tier")=="none", F.lit(None))
                .otherwise(F.initcap("loyalty_tier")),)
            )

parts = F.split("full_name", "", 2)
df_customers = df_customers.withColumn("first_name", parts.getItem(0)).withColumn("last_name",parts.getItem(1))

# step4: dates
date = F.col("registration_date")
df_customers = df_customers.withColumn("registration_date",F.when(date.rlike(r"^\d{4}-\d{2}-\d{2}$"), F.to_date(date,"yyyy-MM-dd"))
                                       .when(date.rlike(r"^\d{2}/\d{2}/\d{4}$"), F.to_date(date,"dd/MM/yyyy"))
                                       .when(date.rlike(r"^\d{2}-\d{2}-\d{4}$"), F.to_date(date,"MM-dd-yyyy")), 
                                       )

print(df_customers.filter(F.col("registration_date").isNull()).count())

#Step 5: booleans
df_customers = df_customers.withColumn("marketing_opt_in", F.when(F.lower("marketing_opt_in").isin("y","1","true"), True)
                                       .when(F.lower("marketing_opt_in").isin("n","0","false"), False),)

df_customers.groupBy("marketing_opt_in").count().show()

#step 6: remove duplicates
meta = ["ingested_at","source_file","batch_id"]
cols = [c for c in df_customers.columns if c not in meta]

print(df_customers.count())
duplicates_by_key =df_customers.dropDuplicates(["customer_id"]).count()
duplicates_by_row =df_customers.dropDuplicates(cols).count()

print("duplicates by key:", duplicates_by_key)
print("duplcates by row:", duplicates_by_row)
assert duplicates_by_key == duplicates_by_row, "Some customers have copies that still differ"

df_customers= df_customers.dropDuplicates(["customer_id"])

df_customers.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema}.customers_silver")


In [0]:
#Orders Transformation
#step 1: moving bad records to bad_records
bronze_columns = ['order_id','customer_id', 'product_id', 'order_date', 'status', 'channel', 'amount', 'quantity', 'priority_flag']
df_orders = df_orders.withColumn("raw records", F.to_json(F.struct(*bronze_columns)))

#Step 2: remove spaces
for c in bronze_columns:
    df_orders = df_orders.withColumn(c, F.when(F.trim(F.col(c)) == "", F.lit(None)).otherwise(F.trim(F.col(c))))

#Step 3: standardize order IDs
distinct_ids_before = df_orders.select('order_id').distinct().count()

df_orders = df_orders.withColumn('order_id_cleaned', F.regexp_replace('order_id', '^[A-Za-z]',''))

improper_id =df_orders.filter(~F.col('order_id_cleaned').rlike(r'^\d+$')).count()
distinct_ids_after = df_orders.select('order_id_cleaned').distinct().count()

assert improper_id == 0,"IDs arent fully cleaned"
assert distinct_ids_before == distinct_ids_after, "Removing the prefix made two different order IDs identical"

df_orders = df_orders.drop('order_id').withColumnRenamed('order_id_cleaned','order_id')

#step 4: standadize status and channel column
status_lower = F.lower(df_orders['status'])
df_orders = df_orders.withColumn("status",
                                F.when(status_lower.isin("c","completed"),"Completed")
                                 .when(status_lower.isin("p","pending"), "Pending")
                                 .when(status_lower.isin("r","returned"), "Returned")
                                 .when(status_lower.isin("x", "cancelled"), "Cancelled")
                                 .otherwise("Unknown")
)

df_orders.select('channel').distinct().show()
channel_lower = F.lower(df_orders['channel'])
df_orders = df_orders.withColumn("channel",
                                F.when(channel_lower == "online","Online")
                                .when(channel_lower == "in-store","In-Store")
                                .when(channel_lower == "b2b", "B2B")
                                .when(channel_lower == "wholesale", "Wholesale")
                                .otherwise("Unknown"))

#step 5: standardaize the date
d = F.col('order_date')
df_orders= df_orders.withColumn('order_date', 
                                F.when(d.rlike(r"^\d{4}-\d{2}-\d{2}$"), F.to_date(d, "yyyy-MM-dd"))
                                .when(d.rlike(r"^\d{2}/\d{2}/\d{4}$"), F.to_date(d, "dd/MM/yyyy"))
                                .when(d.rlike(r"^\d{2}-\d{2}-\d{4}$"), F.to_date(d, "MM-dd-yyyy")))

#step 6 cast amount data types
df_orders = df_orders.withColumn("amount", F.regexp_replace("amount", "R",""))
df_orders = df_orders.withColumn("amount", F.regexp_replace("amount", ',','.'))
df_orders = df_orders.withColumn("amount",F.col("amount").cast("decimal(12,2)"))

df_orders= df_orders.withColumn("quantity", F.col("quantity").cast("int"))

#step 7 standardizing the priority flag column
df_orders = df_orders.withColumn("priority_flag", F.when(F.lower("priority_flag").isin( "0","n","false"), False)
                                 .when(F.lower("priority_flag").isin("1","y","true"), True))
#step 8 remove duplicates
by_key = df_orders.dropDuplicates(["order_id"]).count()
by_row = df_orders.drop("ingested_at", "source_file", "batch_id", "raw_record").dropDuplicates().count()

assert by_key == by_row, "Some orders share the same id but dont have the same informaion"
df_orders = df_orders.dropDuplicates(["order_id"])

#step 9: store bad recoreds
df_customers_silver = spark.table(f"{catalog}.{schema}.customers_silver")
valid_customers = df_customers_silver.select("customer_id")
missing_customers =(df_orders.filter(F.col("customer_id").isNull()))
missing_customers = missing_customers.withColumn("rejection_reason", F.lit("Missing Customer Id"))
orphan_customers = (df_orders.filter(F.col("customer_id").isNotNull())
                    .join(valid_customers, on ="customer_is", how="left_anti")
                    .withColumn("rejection_reason", F.lit("Orphan Customer")))



In [0]:
#Products Trandformation
#step 1: removing unesseccary spaces
text_columns=["product_id","sku","product_name","category","sub_category","description","active_flag"]

for c in text_columns:
    df_products = df_products.withColumn(c, F.when(F.trim(F.col(c)) == "",F.lit(None)).otherwise(F.trim(F.col(c))))

#step 2: fix IDs
df_products =df_products.withColumn("product_id_cleaned", F.regexp_replace("Product_id","^[A-Za-z]", ""))

improper_id = df_products.filter(~F.col("product_id_cleaned").rlike(r"^\d+$")).count()
assert improper_id == 0, "some IDs are not digits only"

df_products= df_products.drop("product_id").withColumnRenamed("product_id_cleaned", "product_id")

#step 3: fix the text 
df_products = (df_products.withColumn("product_name",F.initcap("product_name"))
               .withColumn("category", F.initcap("category"))
               .withColumn("sub_category", F.initcap("sub_category"))
               )

#step 4: standardize prices 
price_columns = ["cost_price","retail_price"]
for c in price_columns:
    df_products= df_products.withColumn(c,F.regexp_replace(c,"R",""))
    df_products =df_products.withColumn(c,F.regexp_replace(c,",","."))
    df_products = df_products.withColumn(c, F.col(c).cast("decimal(12,2)"))

df_products.printSchema()
df_products.filter(F.col("cost_price").isNull()).count()


#step 5: booleans
df_products = df_products.withColumn("active_flag", F.when(F.lower("active_flag").isin("y","1","true"), True)
                                     .when(F.lower("active_flag").isin("n","0","false"), False),)

#step 6: removing duplicates
meta = ["ingested_at","source_file","batch_id"]
cols = [c for c in df_products.columns if c not in meta]
duplicates_by_key = df_products.dropDuplicates(["product_id"]).count()
duplicates_by_row = df_products.dropDuplicates(cols).count()
assert duplicates_by_key == duplicates_by_row, "Some products share the same ID but different content"
df_products = df_products.dropDuplicates(["product_id"])

#step 7: flaffing products that are sold less than what they retail for

df_products = df_products.withColumn("price_below_cost", F.col("retail_price") < F.col("cost_price"))

df_products.show(10)

df_products.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema}.products_silver")



In [0]:
#Transactions Transformation